# PLAY WITH SQL & LLM

In [1]:
from langchain_community.utilities import SQLDatabase
import sqlite3
import os

# set database path
data_folder = "./data"
ddbb_path = os.path.join(data_folder, "recipes.db")

# get database and prompt table names
db = SQLDatabase.from_uri(f"sqlite:///{ddbb_path}")
print(f"database tables are: \n{db.get_usable_table_names()}")

database tables are: 
['cook_technique', 'dish_order', 'ingredient', 'ingredient_category', 'localization', 'recipe', 'recipe_ingredient', 'recipe_ingredient_category']


# MODEL PULLING

Pull or download the `llama3.2:1b` model from the Ollama library.

In [2]:
# download model to fine-tune
import ollama

ollama.pull("llama3.2:1b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

# SIMPLE QUERIES: Ask about numbers

In [3]:
## translate functions
import requests

def correct_response_text(translated_sentences):
    """
    Function that processes a list of translated sentences, correcting formatting issues by merging sentence fragments.
    
    Parameters:
        translated_sentences (list): A list of strings representing translated sentences.
    
    Returns:
        str: A formatted string where certain sentences are merged for better readability.
    """
    corrected_sentences = []
    ctrl = False

    # Iterate over the translated sentences
    for i in range(len(translated_sentences)):
        # Check if the sentence starts with a number (e.g., '1.')
        if translated_sentences[i].strip().endswith('.'):
            # If the next sentence exists, join the current sentence with the next one
            if i + 1 < len(translated_sentences):
                corrected_sentences.append(translated_sentences[i].strip() + ' ' + translated_sentences[i + 1].strip())
                ctrl = True # mark to ignore next sentence as its is joined 
            else:
                corrected_sentences.append(translated_sentences[i].strip())
        elif ctrl:
             ctrl = False # reset control
             continue;
        else:
            # If not starting with a number, just add the sentence as it is
            corrected_sentences.append(translated_sentences[i].strip())
    
    # Join the sentences with a newline character
    result_string = '\n'.join(corrected_sentences)
    return result_string


def translate_elia_session(text, src_lang, dst_lang, verbose = False):
    """
    Function that translates a given text from a source language to a target language using the Elia translation service.

    Parameters:
        text (str): The text to be translated.
        src_lang (str): The source language code (e.g., 'eu' for Basque).
        dst_lang (str): The target language code (e.g., 'en' for English).
        verbose (bool, optional): If True, prints the response JSON for debugging. Default is False.

    Returns:
        str: The translated text after processing.
    """
    # URL of the main translation page (GET request to retrieve CSRF token and cookies)
    url = "https://elia.eus/traductor"  
    
    # URL for making the POST request to get the translated text
    post_url = "https://elia.eus/ajax/translate_string"  

    # Create a session to maintain cookies
    session = requests.Session()

    # Perform the initial GET request to get cookies and the CSRF token
    response = session.get(url)
    
    # Check if the request was successful
    if response.status_code != 200:
        return f"Error fetching the page: {response.status_code}"

    # Extract the CSRF token from the cookies (this can vary depending on the HTML structure)
    csrf_token = None
    for cookie in session.cookies:
        if cookie.name == "csrftoken":  # Look for the csrf token in the cookies
            csrf_token = cookie.value
            break

    # If no CSRF token is found, return an error
    if not csrf_token:
        return "CSRF token not found."

    # Prepare the headers for the POST request, specifying content type, and referring origin
    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",  # Accepting JSON response
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",  # Content type for form submission
        "X-Requested-With": "XMLHttpRequest",  # Indicate the request is an AJAX request
        "Origin": "https://elia.eus",  # Origin header (same as the website)
        "Referer": url,  # Referer header to indicate the source of the request
    }

    # Prepare the payload (data) for the POST request
    data = {
        'csrfmiddlewaretoken': csrf_token,  # CSRF token to prevent cross-site request forgery
        'source_language': src_lang,  # Source language code (e.g., 'eu' for Basque)
        'input_text': text,  # The text to be translated
        'translation_engine': '1',  # The translation engine (usually a default value)
        'target_language': dst_lang,  # Target language code (e.g., 'es' for Spanish)
        'translation_model': 'general',  # The translation model (e.g., 'general' translation)
        'target_voice': 'F',  # Voice for the target language, 'F' for female and 'M' for male
    }

    # Perform the POST request to get the translated text
    response = session.post(post_url, data=data, headers=headers)
    
    # Check if the POST request was successful
    if response.status_code == 200:
        result = response.json()  # Parse the response as json     
        if verbose:
            print(result)
        return correct_response_text(result['translated_sentences']) #get translated sentences and compound string
    else:
        return f"Error translating: {response.status_code}"  # Return an error if translation fails

In [7]:
from langchain.chains import create_sql_query_chain
from langchain.llms import Ollama
import re

def extract_sql_from_response(response):
    pattern = r'```sql\n(SELECT .*?);?\n```'

    # find only sql to execute to database
    codigo_sql = re.search(pattern, response, re.DOTALL)
    
    if codigo_sql:
        return codigo_sql.group(1)
    

# Load Ollama model
llm = Ollama(model="llama3.2:1b")

# prepare question template
term_list = ["sukalde teknika", "plater ordena", "ingrediente edo osagai", "ingrediente edo osagai kategoria", "lokalizazio"]
for term in term_list:
    question_eu = f"Zenbat {term} dituzu? zenbaki osotan"
    print(f"---\n<USER>: {question_eu}")
    
    # create sql query chain
    chain = create_sql_query_chain(llm, db)
    translated_question = translate_elia_session(question_eu, "eu", "en")
    response = chain.invoke({"question": translated_question})
    #print(response)
    if response:
         # get response query from response
        response_sql = extract_sql_from_response(response)
        if response_sql:
            print(f"---\nquery to apply is:{response_sql}")
            
            # database answer is
            result = db.run(response_sql)
            if result:
                resulted_text = f"\nIn total are: {result} {term}."  
                print(f"<MODEL>: {translate_elia_session(resulted_text, 'en', 'eu')}")       

---
<USER>: Zenbat sukalde teknika dituzu? zenbaki osotan
---
query to apply is:SELECT COUNT(DISTINCT cook_technique_id) AS num_cooking_techniques
FROM cook_technique
<MODEL>: Guztira: [(15,)] sukalde teknika.
---
<USER>: Zenbat plater ordena dituzu? zenbaki osotan
---
query to apply is:SELECT COUNT(dish_order_id) AS number_of_orders FROM dish_order
<MODEL>: Guztira: [(3,)] ordenako ordena.
---
<USER>: Zenbat ingrediente edo osagai dituzu? zenbaki osotan
---
query to apply is:SELECT COUNT(ingredient_id) AS num_ingredients,
       COUNT(DISTINCT ingredient_category) + 
       SUM(CASE WHEN category_id IS NOT NULL THEN 1 ELSE 0 END) AS total_num_ingredients
FROM 
   (SELECT DISTINCT ingredient_category FROM ingredient_category) ic
UNION ALL
SELECT COUNT(ingredient_id) AS num_ingredients,
       COUNT(DISTINCT ingredient_category) + 
       SUM(CASE WHEN category_id = 1 THEN 1 ELSE 0 END) AS total_num_ingredients
FROM 
   (SELECT DISTINCT ingredient_category FROM ingredient_category WHERE

OperationalError: (sqlite3.OperationalError) no such column: ingredient_id
[SQL: SELECT COUNT(ingredient_id) AS num_ingredients,
       COUNT(DISTINCT ingredient_category) + 
       SUM(CASE WHEN category_id IS NOT NULL THEN 1 ELSE 0 END) AS total_num_ingredients
FROM 
   (SELECT DISTINCT ingredient_category FROM ingredient_category) ic
UNION ALL
SELECT COUNT(ingredient_id) AS num_ingredients,
       COUNT(DISTINCT ingredient_category) + 
       SUM(CASE WHEN category_id = 1 THEN 1 ELSE 0 END) AS total_num_ingredients
FROM 
   (SELECT DISTINCT ingredient_category FROM ingredient_category WHERE category_id = 1) ic
UNION ALL
SELECT COUNT(ingredient_id) AS num_ingredients,
       COUNT(DISTINCT ingredient_category) + 
       SUM(CASE WHEN category_id = 2 THEN 1 ELSE 0 END) AS total_num_ingredients
FROM 
   (SELECT DISTINCT ingredient_category FROM ingredient_category WHERE category_id = 2) ic
UNION ALL
SELECT COUNT(ingredient_id) AS num_ingredients,
       COUNT(DISTINCT ingredient_catego